# The AI Power Map
## Entrega 4 — Segmentación, Cálculos Analíticos y Fuentes para Tableau
### Equipo: DataChasquiAILab | Semana 11

---

## Introducción

Esta entrega construye sobre el modelo de datos de la Entrega 3 (esquema estrella) 
para generar las métricas derivadas, segmentaciones y tablas analíticas finales 
que alimentarán el dashboard en Tableau (Entrega 5).

El objetivo es que al finalizar esta entrega, Tableau pueda conectarse directamente 
a los CSVs exportados sin ningún reprocesamiento adicional.

### Arquitectura Medallion
- **Bronze** → `data/raw/all_ai_models.csv` — dato crudo original
- **Silver** → `data/processed/all_ai_models_clean.csv` — dato limpio (Entrega 2)
- **Gold** → `outputs/tableau/` — tablas analíticas finales para Tableau (esta entrega)

### Contenido de esta entrega
1. Carga y validación del dataset Silver
2. Segmentación — bloque geopolítico y era tecnológica
3. Métricas derivadas
4. Tablas resumen por segmento
5. Exportación Gold para Tableau
6. Documento de reglas de métricas

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np

from src.config import CLEAN_DATASET, OUTPUTS_TABLEAU
from src.io_utils import load_csv, save_csv
from src.analytics import build_analytical_dataset, build_segment_summaries

print('Librerías e imports listos')
print(f'Proyecto raíz: {PROJECT_ROOT}')

✅ Librerías e imports listos
Proyecto raíz: d:\Proyectos_Code\Data_Visualization_TF


---
## 1. Carga y validación del dataset Silver

Cargamos el dataset limpio generado en la Entrega 2.
Validamos que cumple los requisitos mínimos antes de calcular métricas.

**¿Por qué validar antes de calcular?**
Si el dataset tiene problemas de calidad no detectados, las métricas derivadas
heredarán esos problemas. Validar primero garantiza que los cálculos son confiables.

In [ ]:
df = load_csv(CLEAN_DATASET)

print('='*60)
print('VALIDACIÓN DEL DATASET SILVER')
print('='*60)
print(f'Filas: {df.shape[0]}')
print(f'Columnas: {df.shape[1]}')
print(f'Periodo: {int(df["Year"].min())} – {int(df["Year"].max())}')
print(f'Países únicos: {df["Country (of organization)"].nunique()}')
print(f'Organizaciones únicas: {df["Organization"].nunique()}')
print(f'Dominios únicos: {df["Domain"].nunique()}')
print(f'\nRequisitos mínimos del curso:')
print(f'  ≥ 2,000 registros: {"✅" if df.shape[0] >= 2000 else "❌"} ({df.shape[0]})')
print(f'  ≥ 10 variables útiles: {"✅" if df.shape[1] >= 10 else "❌"} ({df.shape[1]})')
print(f'  Dimensión temporal: ✅ Year, Month')
print(f'  Dimensión geográfica: ✅ Country (of organization)')
print(f'  Dimensión categórica: ✅ Domain, Organization categorization')

VALIDACIÓN DEL DATASET SILVER
Filas: 3405
Columnas: 22
Periodo: 1950 – 2026
Países únicos: 45
Organizaciones únicas: 1306
Dominios únicos: 22

Requisitos mínimos del curso:
  ≥ 2,000 registros: ✅ (3405)
  ≥ 10 variables útiles: ✅ (22)
  Dimensión temporal: ✅ Year, Month
  Dimensión geográfica: ✅ Country (of organization)
  Dimensión categórica: ✅ Domain, Organization categorization


---
## 2. Segmentación

### ¿Qué es segmentación?
Segmentar es dividir el dataset en grupos con características comunes
para poder comparar subgrupos entre sí. Sin segmentación solo podemos
ver el total global — con segmentación podemos responder preguntas como
¿cómo se diferencia Europa de China en producción de modelos de IA?

### Segmentos definidos

**1. Bloque geopolítico**
Agrupa países en bloques según su posición geopolítica en el ecosistema de IA.
Permite comparar potencias, emergentes y rezagados.

| Bloque | Países incluidos |
|---|---|
| Anglosphere | EE.UU., UK, Canadá, Australia |
| China | China, Hong Kong |
| Europa | Alemania, Francia, Suiza, etc. |
| Asia-Pacífico | Corea, Japón, Singapur, India |
| Medio Oriente | Israel, UAE, Arabia Saudita |
| Latinoamérica | Brasil, México, Argentina, etc. |
| Otro | Resto del mundo |

**2. Era tecnológica**
Agrupa modelos según el periodo histórico de la IA en que fueron creados.
Permite análisis longitudinal de cómo evolucionó la producción por época.

| Era | Periodo | Característica |
|---|---|---|
| Era Simbólica | 1950-1979 | IA basada en reglas |
| Era Conexionista | 1980-1999 | Redes neuronales iniciales |
| Era Deep Learning | 2000-2011 | Primeras redes profundas |
| Era GPU | 2012-2016 | Aceleración con GPUs |
| Era Transformer | 2017-2022 | Arquitectura Attention |
| Era LLM | 2023-2026 | Modelos de lenguaje masivos |

In [4]:
# Aplicar todas las transformaciones analíticas
df_gold = build_analytical_dataset(df)

print('='*60)
print('RESULTADO DE SEGMENTACIÓN')
print('='*60)

print(f'\nDataset Gold: {df_gold.shape[0]} filas x {df_gold.shape[1]} columnas')
print(f'Columnas nuevas agregadas: {df_gold.shape[1] - df.shape[1]}')

print(f'\nBloques geopolíticos:')
print(df_gold['bloque_geopolitico'].value_counts().to_string())

print(f'\nEras tecnológicas:')
print(df_gold['era_tecnologica'].value_counts().to_string())

print(f'\nTipo de acceso:')
print(df_gold['tipo_acceso'].value_counts().to_string())

RESULTADO DE SEGMENTACIÓN

Dataset Gold: 3405 filas x 45 columnas
Columnas nuevas agregadas: 23

Bloques geopolíticos:
bloque_geopolitico
Anglosphere      1991
China             847
Europa            232
Asia-Pacífico     167
Otro               81
Medio Oriente      52
Rusia              33
Latinoamérica       2

Eras tecnológicas:
era_tecnologica
Era LLM (2023-2026)              2020
Era Transformer (2017-2022)       921
Era GPU (2012-2016)               231
Era Deep Learning (2000-2011)     135
Era Conexionista (1980-1999)       68
Era Simbólica (1950-1979)          30

Tipo de acceso:
tipo_acceso
Cerrado    1305
Abierto    1303
Unknown     797


---
## 3. Métricas derivadas

### ¿Qué son métricas derivadas?
Son variables nuevas calculadas a partir de las existentes que responden
preguntas analíticas específicas. No estaban en el dataset original —
las construimos nosotros con un propósito claro.

### Métricas calculadas

| Métrica | Fórmula | Para qué sirve en Tableau |
|---|---|---|
| `log_parameters` | log(Parameters + 1) | Escala logarítmica para scatter plot |
| `log_training_compute_(flop)` | log(Compute + 1) | Escala logarítmica para serie temporal |
| `log_citations` | log(Citations + 1) | Escala logarítmica para comparación |
| `modelos_anio` | COUNT por Year | Serie temporal de producción anual |
| `modelos_acumulados_global` | CUMSUM por Year | Crecimiento acumulado global |
| `modelos_pais` | COUNT por Country | Ranking de países |
| `pct_participacion_pais` | modelos_pais / total * 100 | Participación porcentual por país |
| `modelos_dominio` | COUNT por Domain | Ranking de dominios |
| `pct_participacion_dominio` | modelos_dominio / total * 100 | Participación por dominio |

**¿Por qué logaritmo y no escala lineal?**
Parameters va de 10 a 173 billones — escala lineal aplana todos los valores
pequeños contra los gigantes. El logaritmo preserva las proporciones relativas
y hace el gráfico legible para toda la distribución.

In [5]:
print('='*60)
print('VERIFICACIÓN DE MÉTRICAS DERIVADAS')
print('='*60)

metricas_nuevas = [
    'bloque_geopolitico', 'era_tecnologica', 'tipo_acceso',
    'log_parameters', 'log_training_compute_(flop)', 'log_citations',
    'modelos_anio', 'modelos_acumulados_global',
    'modelos_pais', 'pct_participacion_pais',
    'modelos_dominio', 'pct_participacion_dominio'
]

for col in metricas_nuevas:
    if col in df_gold.columns:
        nulos = df_gold[col].isnull().sum()
        print(f'  ✅ {col}: {nulos} nulos')
    else:
        print(f'  ❌ {col}: NO ENCONTRADA')

print(f'\nEstadísticas de métricas numéricas clave:')
cols_stats = ['modelos_anio', 'modelos_acumulados_global', 
              'pct_participacion_pais', 'pct_participacion_dominio']
print(df_gold[cols_stats].describe().round(2).to_string())

VERIFICACIÓN DE MÉTRICAS DERIVADAS
  ✅ bloque_geopolitico: 0 nulos
  ✅ era_tecnologica: 0 nulos
  ✅ tipo_acceso: 0 nulos
  ✅ log_parameters: 1162 nulos
  ❌ log_training_compute_(flop): NO ENCONTRADA
  ✅ log_citations: 1943 nulos
  ✅ modelos_anio: 0 nulos
  ✅ modelos_acumulados_global: 0 nulos
  ✅ modelos_pais: 0 nulos
  ✅ pct_participacion_pais: 0 nulos
  ✅ modelos_dominio: 0 nulos
  ✅ pct_participacion_dominio: 0 nulos

Estadísticas de métricas numéricas clave:
       modelos_anio  modelos_acumulados_global  pct_participacion_pais  pct_participacion_dominio
count       3405.00                    3405.00                 3405.00                    3405.00
mean         467.75                    1936.37                   31.12                      26.98
std          328.59                    1101.26                   20.20                      20.96
min            1.00                       1.00                    0.03                       0.03
25%          168.00                     947

---
## 4. Tablas resumen por segmento

Generamos tablas agregadas por cada segmento para Tableau.
Estas tablas son más livianas que el dataset completo y permiten
vistas rápidas sin que Tableau procese 3,405 filas cada vez.

In [6]:
summaries = build_segment_summaries(df_gold)

print('='*60)
print('TABLAS RESUMEN GENERADAS')
print('='*60)

for nombre, tabla in summaries.items():
    print(f'\n{nombre}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas')
    print(tabla.head(3).to_string(index=False))

TABLAS RESUMEN GENERADAS

seg_pais_anio: 321 filas x 4 columnas
Country (of organization) bloque_geopolitico   Year  modelos
                Argentina      Latinoamérica 2013.0        1
                Australia        Anglosphere 2016.0        2
                Australia        Anglosphere 2017.0        1

seg_bloque_era: 35 filas x 3 columnas
bloque_geopolitico               era_tecnologica  modelos
       Anglosphere  Era Conexionista (1980-1999)       53
       Anglosphere Era Deep Learning (2000-2011)      103
       Anglosphere           Era GPU (2012-2016)      181

seg_dominio_era: 89 filas x 3 columnas
     Domain               era_tecnologica  modelos
3D modeling Era Deep Learning (2000-2011)        1
3D modeling           Era GPU (2012-2016)        1
3D modeling           Era LLM (2023-2026)       13

seg_org_era: 19 filas x 3 columnas
Organization categorization               era_tecnologica  modelos
                   Academia  Era Conexionista (1980-1999)       46
       

---
## 5. Exportación Gold para Tableau

Exportamos todas las tablas a `outputs/tableau/`.
Estas son las fuentes finales que conectaremos en Tableau en la Entrega 5.

### Tablas exportadas

| Archivo | Contenido | Uso en Tableau |
|---|---|---|
| `fact_models_gold.csv` | Dataset completo con todas las métricas | Fuente principal |
| `seg_pais_anio.csv` | Modelos por país y año | Serie temporal por país |
| `seg_bloque_era.csv` | Modelos por bloque geopolítico y era | Comparación geopolítica |
| `seg_dominio_era.csv` | Modelos por dominio y era | Evolución por dominio |
| `seg_org_era.csv` | Modelos por tipo de organización y era | Academia vs Industria |
| `seg_acceso_era.csv` | Modelos por accesibilidad y era | Evolución de apertura |

In [7]:
# Exportar dataset Gold principal
save_csv(df_gold, OUTPUTS_TABLEAU / 'fact_models_gold.csv')
print(f'✅ fact_models_gold.csv exportado: {df_gold.shape[0]} filas x {df_gold.shape[1]} columnas')

# Exportar tablas resumen
for nombre, tabla in summaries.items():
    save_csv(tabla, OUTPUTS_TABLEAU / f'{nombre}.csv')
    print(f'✅ {nombre}.csv exportado: {tabla.shape[0]} filas')

print(f'\n✅ Todos los archivos exportados en: {OUTPUTS_TABLEAU}')

✅ fact_models_gold.csv exportado: 3405 filas x 45 columnas
✅ seg_pais_anio.csv exportado: 321 filas
✅ seg_bloque_era.csv exportado: 35 filas
✅ seg_dominio_era.csv exportado: 89 filas
✅ seg_org_era.csv exportado: 19 filas
✅ seg_acceso_era.csv exportado: 16 filas

✅ Todos los archivos exportados en: D:\Proyectos_Code\Data_Visualization_TF\outputs\tableau


---
## 6. Documento de reglas de métricas, segmentos y parámetros

### Reglas de métricas

**pct_participacion_pais**
- Fórmula: modelos del país / total de modelos × 100
- Denominador: 3,405 modelos totales en el dataset
- Interpretación: qué porcentaje de la producción global corresponde a ese país
- Limitación: no pondera por tamaño del modelo ni por impacto

**modelos_acumulados_global**
- Fórmula: suma acumulada de modelos por año ordenado cronológicamente
- Interpretación: stock total de modelos producidos hasta ese año
- Uso en Tableau: eje Y en vista longitudinal

**log_parameters**
- Fórmula: log(Parameters + 1)
- Por qué +1: evita log(0) para modelos con Parameters = 0
- Interpretación: cada unidad representa un orden de magnitud
- Uso en Tableau: scatter plot de tamaño vs cómputo

### Segmentos definidos

**bloque_geopolitico**
- Criterio: país de la organización mapeado a bloque
- Países sin mapeo explícito → 'Otro'
- Uso en Tableau: filtro y dimensión de color en comparaciones

**era_tecnologica**
- Criterio: año de publicación del modelo
- Modelos sin año → 'Desconocido'
- Uso en Tableau: dimensión para análisis longitudinal por periodo

### Parámetros sugeridos para Tableau
- Filtro por bloque_geopolitico → permite comparar regiones
- Filtro por era_tecnologica → permite ver evolución por periodo
- Filtro por Domain → permite ver por tipo de IA
- Filtro por tipo_acceso → permite comparar abierto vs cerrado